In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import cvzone

In [ ]:
model = YOLO("C://Users//Asus//Downloads//best.pt")
class_names = model.names
cap = cv2.VideoCapture("C://Users//Asus//Downloads//pothole.mp4")
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
count = 0

In [ ]:
# Solution 2: Trying this method by not resizing the frame and Generating region of interest 
count = 0
pothole_areas = []

while True:
    ret, img = cap.read()
    if not ret:
        break
    count += 1
    if count % 3 != 0:
        continue

    # Resize the image
    img = cv2.resize(img, (1020, 500))

    # Predict using the model on the entire image
    results = model.predict(img)

    for r in results:
        boxes = r.boxes
        masks = r.masks

    if masks is not None:
        masks = masks.data.cpu()
        for seg, box in zip(masks.data.cpu().numpy(), boxes):
            seg = cv2.resize(seg, (1020, 500))  # Resize to match the full image size
            contours, _ = cv2.findContours((seg).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for contour in contours:
                d = int(box.cls)
                c = class_names[d]
                x, y, w, h = cv2.boundingRect(contour)

                # generating ROI 
                if y + h > 3 * 500 // 5:
                    area = cv2.contourArea(contour)
                    pothole_areas.append((c, int(area)))

                    # Choose color based on area
                    color = (0, 0, 255) if area > 10000 else (255, 0, 0)

                    # Draw contours with the chosen color
                    cv2.polylines(img, [contour], True, color=color, thickness=2)
                    cv2.putText(img, c, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    # Display the image with detections
    cv2.imshow('img', img)
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

    for area in pothole_areas:
        print(f'Detected pothole with area: {area} px')

cap.release()
cv2.destroyAllWindows()